# 🧠 Sigmoid Perceptron — Theory & Math Notebook
### A complete guide to understanding the Perceptron from scratch

---

## 📚 Table of Contents
1. [What is a Perceptron?](#1-what-is-a-perceptron)
2. [The Neuron Analogy](#2-the-neuron-analogy)
3. [Weights & Bias](#3-weights--bias)
4. [The Weighted Sum (Z)](#4-the-weighted-sum-z)
5. [The Sigmoid Activation Function](#5-the-sigmoid-activation-function)
6. [Making a Prediction](#6-making-a-prediction)
7. [Measuring Error / Loss](#7-measuring-error--loss)
8. [The Derivative of Sigmoid](#8-the-derivative-of-sigmoid)
9. [Gradient Descent — How the Brain Learns](#9-gradient-descent--how-the-brain-learns)
10. [Weight & Bias Update Rules](#10-weight--bias-update-rules)
11. [Training Loop — Epochs](#11-training-loop--epochs)
12. [Data Preprocessing](#12-data-preprocessing)
13. [StandardScaler — Feature Scaling](#13-standardscaler--feature-scaling)
14. [Train Test Split](#14-train-test-split)
15. [Model Evaluation — Accuracy](#15-model-evaluation--accuracy)
16. [Full Flow Summary](#16-full-flow-summary)

---

## 1. What is a Perceptron?

A **Perceptron** is the simplest unit of a Neural Network. It is a mathematical model inspired by how a single neuron works in the human brain.

It takes multiple **inputs**, multiplies each by a **weight**, adds them all up, adds a **bias**, and passes the result through an **activation function** to produce an **output**.

> Think of it as a tiny brain cell that takes information, processes it, and makes a YES or NO decision.

A Sigmoid Perceptron specifically uses the **Sigmoid function** as its activation, which means it outputs a **probability between 0 and 1**.

---

## 2. The Neuron Analogy

```
Real Brain Neuron:                    Perceptron:
                                      
  Signal 1 ──(strength)──┐            x1 ──(w1)──┐
  Signal 2 ──(strength)──┤──[NEURON]  x2 ──(w2)──┤──[Σ + bias]──[sigmoid]──► output
  Signal 3 ──(strength)──┘            x3 ──(w3)──┘
```

| Brain Neuron | Perceptron |
|---|---|
| Input signals | Input features (age, bmi, glucose...) |
| Signal strength | Weights |
| Neuron fires or not | Sigmoid output (0 to 1) |
| Threshold | Bias |

---

## 3. Weights & Bias

### Weights (w)

Each input feature gets its own weight. The weight controls **how important** that feature is to the final decision.

- High weight → feature matters a LOT
- Low weight → feature barely matters
- Negative weight → feature pulls the decision the opposite way

At the start, weights are initialized **randomly** using a Normal Distribution:

$$w = \mathcal{N}(0, 1)$$

In code:
```python
self.weights = np.random.randn(input_size)
```

### Bias (b)

The bias is a single extra number that gives the model a **default lean** before even looking at inputs. Like a doctor who slightly suspects diabetes before examining a patient.

$$b = \mathcal{N}(0, 1)$$

In code:
```python
self.bias = np.random.randn(1)
```

---

## 4. The Weighted Sum (Z)

This is the core calculation of every neural network. Each input is multiplied by its weight, all are added together, and the bias is added at the end.

### Formula

$$z = (x_1 \cdot w_1) + (x_2 \cdot w_2) + (x_3 \cdot w_3) + \ldots + (x_n \cdot w_n) + b$$

In compact notation using the **dot product**:

$$z = \mathbf{x} \cdot \mathbf{w} + b$$

Where:
- $\mathbf{x}$ = input vector (all features of one patient)
- $\mathbf{w}$ = weight vector (all weights)
- $b$ = bias

### Example with Diabetes Data

| Feature | Value (x) | Weight (w) | x × w |
|---|---|---|---|
| age | 55 | 0.3 | 16.5 |
| bmi | 27.3 | 0.8 | 21.84 |
| blood_glucose | 140 | 1.2 | 168.0 |
| hypertension | 0 | -0.5 | 0.0 |
| ... | ... | ... | ... |
| **bias** | — | — | **-0.4** |
| **z (total)** | | | **206.0** |

In code:
```python
weighted_sum = np.dot(inputs, self.weights) + self.bias
```

The `z` value can be ANY number — negative, positive, huge, small. That's why we need the sigmoid function to squeeze it into a usable range!

---

## 5. The Sigmoid Activation Function

The sigmoid function takes ANY number and squishes it into a value **strictly between 0 and 1**. This makes the output interpretable as a **probability**.

### Formula

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Where $e \approx 2.71828$ (Euler's number)

### Behavior

| z value | σ(z) output | Meaning |
|---|---|---|
| very large (+∞) | → 1.0 | 100% confident YES |
| +2 | 0.88 | 88% confident YES |
| 0 | 0.50 | 50/50, completely unsure |
| -2 | 0.12 | 88% confident NO |
| very small (-∞) | → 0.0 | 100% confident NO |

### The S-Curve (Bell Curve Shape)

```
Output
  1 |                    ___________
    |                   /
 0.5|.................../............  ← threshold line
    |                 /
  0 |________________/
    +--------------------------------→ z
   -5    -2    0    2    5
```

### Why Sigmoid and Not Just Raw Z?

- Raw z can be -9999 or +50000 → not a probability!
- Sigmoid keeps everything between 0 and 1 → clean probability!
- The curve is smooth and **differentiable** → we can calculate gradients for learning!

In code:
```python
def sigmoid(self, Z):
    return 1 / (1 + np.exp(-Z))
```

---

## 6. Making a Prediction

A prediction is simply running an input through the weighted sum and then the sigmoid:

$$\hat{y} = \sigma(\mathbf{x} \cdot \mathbf{w} + b)$$

Where $\hat{y}$ (y-hat) = the predicted probability (0 to 1)

```
Input Features → [Weighted Sum] → z → [Sigmoid] → probability (0 to 1)
```

In code:
```python
def predict(self, inputs):
    weighted_sum = np.dot(inputs, self.weights) + self.bias
    return self.sigmoid(weighted_sum)
```

---

## 7. Measuring Error / Loss

After predicting, we need to measure **how wrong** we were. This is called the **error** or **loss**.

### Error Formula

$$\text{error} = y - \hat{y}$$

Where:
- $y$ = actual target (0 or 1, the real answer)
- $\hat{y}$ = predicted value (the model's guess, between 0 and 1)

### What the Error Tells Us

| Actual (y) | Predicted (ŷ) | Error | Meaning |
|---|---|---|---|
| 1 | 0.3 | +0.7 | Too low! Need to predict higher |
| 0 | 0.8 | -0.8 | Too high! Need to predict lower |
| 1 | 0.95 | +0.05 | Almost right! Tiny correction needed |
| 0 | 0.02 | -0.02 | Almost right! Tiny correction needed |

In code:
```python
error = target - prediction
```

---

## 8. The Derivative of Sigmoid

To update weights intelligently, we need to know **how steep** the sigmoid curve is at the current prediction point. This is the **derivative**.

### Sigmoid Derivative Formula

$$\frac{d\sigma}{dz} = \sigma(z) \cdot (1 - \sigma(z)) = \hat{y} \cdot (1 - \hat{y})$$

This is one of the most beautiful results in calculus — the sigmoid's own derivative is expressed using the sigmoid itself!

### What This Means

| Prediction (ŷ) | ŷ × (1 - ŷ) | Meaning |
|---|---|---|
| 0.5 | 0.5 × 0.5 = **0.25** | Maximum steepness → learn FAST |
| 0.9 | 0.9 × 0.1 = **0.09** | Less steep → learn moderately |
| 0.1 | 0.1 × 0.9 = **0.09** | Less steep → learn moderately |
| 0.99 | 0.99 × 0.01 = **0.0099** | Almost flat → learn very slowly |

> **Key insight:** When the model is very confident (near 0 or 1), it learns slowly. When it is unsure (near 0.5), it learns fast. This is mathematically perfect behavior!

---

## 9. Gradient Descent — How the Brain Learns

Gradient Descent is the algorithm that makes the perceptron learn. It adjusts weights and bias step by step to reduce error over time.

### The Intuition

Imagine you are blindfolded on a hilly landscape and you want to reach the lowest valley (minimum error). You feel the slope under your feet and take a small step downhill. Repeat thousands of times → you reach the bottom!

```
Error
  ↑
  |  \
  |   \         ← steep slope → take big step
  |    \
  |     \___
  |         \____ ← flat slope → take small step
  |              \___________
  +--------------------------------→ weight value
                      ↑
                   minimum error (goal!)
```

### The Gradient

The gradient tells us the direction and size of the slope at our current position:

$$\text{gradient} = \text{error} \times \hat{y} \times (1 - \hat{y})$$

- **error** → how wrong we are (direction to move)
- **ŷ × (1 - ŷ)** → how steep the curve is (how big a step to take)

---

## 10. Weight & Bias Update Rules

These are the core learning equations derived from **Backpropagation** using the **Chain Rule** of calculus.

### Weight Update

$$\Delta w = \alpha \cdot \text{error} \cdot \hat{y} \cdot (1 - \hat{y}) \cdot x$$

$$w_{\text{new}} = w_{\text{old}} + \Delta w$$

### Bias Update

$$\Delta b = \alpha \cdot \text{error} \cdot \hat{y} \cdot (1 - \hat{y})$$

$$b_{\text{new}} = b_{\text{old}} + \Delta b$$

Where $\alpha$ = **learning rate** (how big each step is)

### Why No Input (x) in Bias Update?

- Weights are **connected** to specific inputs → the adjustment depends on how big that input was
- Bias is **standalone** → not connected to any input → no x needed

### Learning Rate (α)

| Learning Rate | Effect |
|---|---|
| Too large (e.g. 10) | Overshoots the minimum, bounces around, never converges |
| Too small (e.g. 0.000001) | Takes forever to learn |
| Just right (e.g. 0.1) | Smooth, steady learning → reaches minimum nicely |

In code:
```python
# Weights
gradient_weights = error * prediction * (1 - prediction) * input_vector
self.weights += learning_rate * gradient_weights

# Bias
gradient_bias = error * prediction * (1 - prediction)
self.bias += learning_rate * gradient_bias
```

---

## 11. Training Loop — Epochs

### What is an Epoch?

One **epoch** = one complete pass through ALL training examples.

### Why Multiple Epochs?

One pass is not enough to learn well. Like reading a textbook once vs reading it 100 times — each read reinforces understanding!

### The Full Training Loop

```
For each epoch (1 to 100):
    For each patient (1 to 13,736):
        1. Get patient's features (input_vector)
        2. Get real answer (target)
        3. Predict → ŷ = σ(x·w + b)
        4. Calculate error = y - ŷ
        5. Calculate gradients
        6. Update weights
        7. Update bias
    Print epoch progress
```

Total weight updates = 100 epochs × 13,736 patients = **1,373,600 updates!**

In code:
```python
for epoch in range(num_epochs):
    for i in range(num_examples):
        input_vector = inputs[i]
        target = targets[i]
        prediction = self.predict(input_vector)
        error = target - prediction

        gradient_weights = error * prediction * (1 - prediction) * input_vector
        self.weights += learning_rate * gradient_weights

        gradient_bias = error * prediction * (1 - prediction)
        self.bias += learning_rate * gradient_bias

    print(f"Epoch {epoch+1}/{num_epochs} done!")
```

---

## 12. Data Preprocessing

### Why Preprocess?

Raw data is messy. Computers only understand numbers. Text columns must be converted and missing values must be filled.

### Encoding Text to Numbers

The model cannot understand "Male" or "Female" — only numbers!

```python
# Gender encoding
df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})

# Smoking history encoding
df['smoking_history'] = df['smoking_history'].map({
    'never': 0,
    'current': 1,
    'former': 2,
    'No Info': 3
})
```

### Handling Missing Values

```python
df['smoking_history'] = df['smoking_history'].fillna('10')
# fills any blank cells with a placeholder value
```

### Balancing the Dataset

Original data: 92,407 no-diabetes vs 8,585 diabetes → very unbalanced!

If we train on unbalanced data, the model can just say "NO DIABETES" for everyone and get 92% accuracy — but it's useless!

**Solution:** Randomly sample 8,585 from the majority class so both classes are equal:

$$\text{balanced data} = 8585 \text{ (class 0)} + 8585 \text{ (class 1)} = 17170 \text{ total}$$

```python
class_0_df = class_0_df.sample(8585)
data = pd.concat([class_0_df, class_1_df])
```

---

## 13. StandardScaler — Feature Scaling

### The Problem Without Scaling

Your features have very different ranges:

| Feature | Range |
|---|---|
| age | 1 to 80 |
| blood_glucose_level | 80 to 300 |
| hypertension | 0 or 1 |
| bmi | 10 to 70 |

Without scaling, `blood_glucose` dominates because its numbers are huge. The weights for glucose would need to be tiny while weights for hypertension would be huge. Training becomes unstable!

### The Formula

StandardScaler transforms each feature using:

$$x_{\text{scaled}} = \frac{x - \mu}{\sigma}$$

Where:
- $\mu$ = mean of the feature
- $\sigma$ = standard deviation of the feature

### What This Does

After scaling, every feature has:
- Mean = 0
- Standard Deviation = 1
- Most values fall between -3 and +3

All features now play on the **same level field**!

### fit_transform vs transform

```python
# On training data — LEARN the mean & std, then scale
X_train_scaled = scaler.fit_transform(X_train)

# On test data — USE the SAME mean & std from training, just scale
X_test_scaled = scaler.transform(X_test)
```

> **CRITICAL RULE:** Never fit on test data! If you learn the mean from test data, you are "peeking" at the test — that's cheating and will give falsely optimistic results!

---

## 14. Train Test Split

### Why Split Data?

We need to test the model on data it has **NEVER seen before** to get an honest measure of how good it is.

$$\text{Total Data (17,170)} \rightarrow \begin{cases} \text{Train: } 80\% = 13,736 \text{ patients} \\ \text{Test: } 20\% = 3,434 \text{ patients} \end{cases}$$

```python
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.2,        # 20% for testing
    stratify=Y,           # keep equal 0s and 1s in both splits
    random_state=42       # same split every run (reproducibility)
)
```

### stratify=Y

Without stratify, the random split might put most diabetic patients in one group. With stratify, **both train and test maintain the same 50/50 ratio** of diabetes vs no-diabetes.

---

## 15. Model Evaluation — Accuracy

### Formula

$$\text{Accuracy} = \frac{\text{Number of Correct Predictions}}{\text{Total Number of Predictions}} \times 100\%$$

### Decision Threshold

The model outputs a probability (0 to 1). We convert to class using a threshold:

$$\text{predicted\_class} = \begin{cases} 1 & \text{if } \hat{y} \geq 0.5 \\ 0 & \text{if } \hat{y} < 0.5 \end{cases}$$

In code:
```python
def evaluate(self, inputs, targets):
    correct = 0
    for input_vector, target in zip(inputs, targets):
        prediction = self.predict(input_vector)
        if prediction >= 0.5:
            predicted_class = 1
        else:
            predicted_class = 0
        if predicted_class == target:
            correct += 1
    accuracy = correct / len(inputs)
    return accuracy
```

### Result

> **Training Accuracy = 88.64%**
>
> Out of 100 patients, the model correctly diagnosed ~89 of them using nothing but math and Python!

---

## 16. Full Flow Summary

```
RAW CSV (100,992 patients)
         ↓
ENCODE text → numbers (gender, smoking_history)
         ↓
BALANCE dataset (8,585 vs 8,585)
         ↓
SPLIT features (X) and labels (Y)
         ↓
TRAIN/TEST SPLIT (80% / 20%)
         ↓
SCALE features (StandardScaler)
         ↓
CREATE Perceptron (random weights + bias)
         ↓
TRAIN for 100 epochs:
    For each patient:
        predict → error → gradient → update weights & bias
         ↓
EVALUATE → Accuracy = 88.64% 🎉
```

### The 3 Core Math Equations To Remember

**1. Forward Pass (Prediction):**
$$\hat{y} = \sigma(\mathbf{x} \cdot \mathbf{w} + b) = \frac{1}{1 + e^{-(\mathbf{x} \cdot \mathbf{w} + b)}}$$

**2. Error:**
$$\text{error} = y - \hat{y}$$

**3. Backward Pass (Learning):**
$$w_{\text{new}} = w_{\text{old}} + \alpha \cdot \text{error} \cdot \hat{y}(1-\hat{y}) \cdot x$$
$$b_{\text{new}} = b_{\text{old}} + \alpha \cdot \text{error} \cdot \hat{y}(1-\hat{y})$$

---

> 🔥 **Final Note:**
> Every massive AI model in the world — GPT, Gemini, LLaMA — is built from millions of units doing exactly these same three steps. You have understood the true foundation of Deep Learning. Everything from here is just scale!